# Target-server compromise over time on the attack graph
For each defender, the fraction of the attacker's valued servers that are compromised at each step, averaged over 200 pre-drawn episodes against the trained attacker with $\lambda = 0$ (a fully rational attacker). Every defender samples its action from its own distribution.

In [ ]:
%run Initialize.ipynb

## Settings and trained models

In [ ]:
episode_length = 50
val_episodes = 200
LAM_EVAL = 0.0
SAVE_DIR = "compared"

Context_Model = Load(TSTransformer().to(device), "encoder.pth")
critic        = Load(Critic().to(device), "critic.pth")
Player1       = Load(DefenderActor().to(device), "player1.pth")
Player2       = Load(AttackerActor().to(device), "player2.pth")
BASELINES     = Load_Baselines(SAVE_DIR)

val_cfg_rng = np.random.RandomState(54321)
val_episode_configs = []
for _ in range(val_episodes):
    p = val_cfg_rng.permutation(TERMINALS)
    s1_parent, s2_parent = int(p[0]), int(p[1])
    t0 = val_cfg_rng.randint(0, 2)
    t1 = val_cfg_rng.randint(1 - t0, 2)
    x0 = val_cfg_rng.randint(n)
    val_episode_configs.append((s1_parent, s2_parent, t0, t1, x0))

## Rollout

In [ ]:
def Compromise_Over_Time(Defender):
    defw = get_defender(Defender)
    all_srv = []
    for i_episode in tqdm(range(val_episodes), desc=Defender, leave=False):
        np.random.seed(2000 + i_episode); torch.manual_seed(2000 + i_episode)
        random.seed(2000 + i_episode)                      # Next_State draws from Python's random
        s1_parent, s2_parent, t0, t1, x0 = val_episode_configs[i_episode]
        lam = LAM_EVAL
        Theta, Type_Theta, D, rho = Context_From_Config(s1_parent, s2_parent, t0, t1, lam)
        defw.reset(Theta, Type_Theta, lam, x0, D, rho)
        tt_t = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        x = x0
        m2 = np.full(d, 0.2)
        counts = np.zeros(d)
        w = float(t0 + t1)
        ep_srv = [(t0 * State_Matrix[x, 8] + t1 * State_Matrix[x, 9]) / w]
        for t in range(episode_length):
            x_bits = State_Matrix[x, :]
            a_dist = defw.act_dist(x_bits, m2)
            if np.random.rand() < lam:
                u = int(np.random.choice(n_u))
            else:
                with torch.no_grad():
                    u_dist = Player2(torch.tensor(x_bits, dtype=torch.float32, device=device),
                                     torch.tensor(m2, dtype=torch.float32, device=device), tt_t).cpu().numpy()
                u = int(np.random.choice(n_u, p=u_dist / u_dist.sum()))
            a = int(np.random.choice(n_a, p=a_dist))       # the defender samples from its own policy
            next_x = Next_State(x, a, u, D, rho)
            bits = State_Matrix[next_x, :]
            ep_srv.append((t0 * bits[8] + t1 * bits[9]) / w)
            m2, counts = m2_update(x_bits, bits, m2, counts)
            y = Observation(bits)
            defw.observe(a, x_bits, bits, y)
            x = next_x
        all_srv.append(ep_srv)
    mat = np.array(all_srv)
    return mat.mean(axis=0), mat.std(axis=0, ddof=1) / np.sqrt(val_episodes)


VAL_DEFENDERS = ['Critic P1', 'Trained P1', 'RPPO', 'DR_RPPO', 'VARIBAD', 'PEARL', 'MMD']
results_time = {Defender: Compromise_Over_Time(Defender) for Defender in VAL_DEFENDERS}

## Plot

In [ ]:
steps = np.arange(episode_length + 1)
fig, ax = plt.subplots(figsize=(6, 4.5))
for Defender in VAL_DEFENDERS:
    mean, se = results_time[Defender]
    ax.plot(steps, mean, label=Defender)
    ax.fill_between(steps, mean - se, mean + se, alpha=0.15)
ax.set_xlabel('Time', fontsize=14)
ax.set_ylabel('Target server compromise rate', fontsize=14)
ax.set_xlim(0, episode_length)
ax.set_ylim(0, 1)
ax.grid(linestyle='--', linewidth=0.6, alpha=0.5)
ax.set_axisbelow(True)
ax.tick_params(axis='both', labelsize=13)
ax.legend(ncol=4, fontsize=9.8, loc='lower left')
plt.tight_layout()
plt.savefig('Result1.jpg', dpi=300, bbox_inches='tight')
plt.show()